In [4]:
from google.colab import drive
import pandas as pd
mpesa_transactions_anomaly = drive.mount("/content/drive/")
mpesa_transactions_anomaly = "/content/drive/MyDrive/mpesa_synthetic.csv"
try:
    mpesa_transactions_anomaly = pd.read_csv(mpesa_transactions_anomaly)
    print("Success reading mpesa_transactions_anomaly")
except Exception as e:
    print(type(e).__name__, "-", str(e))

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).
Success reading mpesa_transactions_anomaly


In [6]:
print("Rows loaded:", len(mpesa_transactions_anomaly))
print(mpesa_transactions_anomaly.head())

Rows loaded: 120000
  transaction_id   amount  sender_balance_before  sender_balance_after  \
0      UA0IFD0TV   703.90               58586.32              57882.42   
1      UJAHXTHV3   254.44                8088.00               7833.56   
2      UEF8MDD4V   609.04               56675.00              56065.96   
3      UBT3W5UZB  5255.34               75090.36              69835.02   
4      UGKWNNHJ7  7282.67               24408.96              17126.29   

   receiver_balance_before  receiver_balance_after transaction_type  hour  \
0                 29932.92                30636.82             peer    14   
1                 22962.44                23216.88             peer    18   
2                  1029.22                 1638.26             till     7   
3                    38.94                 5294.28          paybill    11   
4                 26237.82                33520.49             till     0   

   month_2026 day_of_week device_type   region  is_fraud  
0           1

In [8]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
df = mpesa_transactions_anomaly

numeric_cols = [
    'amount', 
    'sender_balance_before', 
    'sender_balance_after',
    'receiver_balance_before', 
    'receiver_balance_after', 
    'hour'
]
cat_cols = ['transaction_type', 'device_type', 'region', 'day_of_week']

df = df.sort_values(by=['month_2026', 'hour']).reset_index(drop=True)

X = pd.get_dummies(df[numeric_cols + cat_cols], columns=cat_cols, drop_first=True, dtype=float)
y = df['is_fraud'].astype(bool).to_numpy()

print(f"Total features after encoding: {X.shape[1]}")
print(f"Total fraud transactions: {y.sum()} ({y.sum()/len(y)*100:.2f}%)")

n = len(X)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

X_train, y_train = X.iloc[:train_end], y[:train_end]
X_val, y_val = X.iloc[train_end:val_end], y[train_end:val_end]
X_test, y_test = X.iloc[val_end:], y[val_end:]

X_train_normal = X_train[~y_train]

scaler = StandardScaler()
X_train_norm_scaled = scaler.fit_transform(X_train_normal)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print(f"Train (normal only): {X_train_norm_scaled.shape[0]} rows")
print(f"Validation: {len(X_val)} rows (Fraud: {y_val.sum()})")
print(f"Test: {len(X_test)} rows (Fraud: {y_test.sum()})")

Total features after encoding: 19
Total fraud transactions: 3510 (2.93%)
Train (normal only): 81529 rows
Validation: 18000 rows (Fraud: 515)
Test: 18000 rows (Fraud: 524)


In [11]:
from sklearn.decomposition import PCA
from sklearn.metrics import precision_recall_curve, precision_score, recall_score, f1_score

pca = PCA(n_components=0.95, svd_solver='full')
pca.fit(X_train_norm_scaled)
print(f"PCA retained components: {pca.n_components_} out of {X.shape[1]}")

def get_pca_reconstruction_error(scaled_data):
    reconstructed = pca.inverse_transform(pca.transform(scaled_data))
    return np.mean((scaled_data - reconstructed) ** 2, axis=1)

val_pca_errors = get_pca_reconstruction_error(X_val_scaled)
test_pca_errors = get_pca_reconstruction_error(X_test_scaled)

prec, rec, thresh = precision_recall_curve(y_val, val_pca_errors)
f1_scores = 2 * (prec * rec) / (prec + rec + 1e-12)
best_pca_thresh = thresh[np.argmax(f1_scores)]

test_pca_preds = test_pca_errors >= best_pca_thresh

pca_prec = precision_score(y_test, test_pca_preds, zero_division=0)
pca_rec = recall_score(y_test, test_pca_preds, zero_division=0)
pca_f1 = f1_score(y_test, test_pca_preds, zero_division=0)

print("PCA Anomaly Detection ===")
print(f"Selected Threshold: {best_pca_thresh:.4f}")
print(f"Test Precision:     {pca_prec:.4f}")
print(f"Test Recall:        {pca_rec:.4f}")
print(f"Test F1 Score:      {pca_f1:.4f}")

PCA retained components: 14 out of 19
PCA Anomaly Detection ===
Selected Threshold: 0.0184
Test Precision:     0.0318
Test Recall:        0.7462
Test F1 Score:      0.0610


In [ ]:
from sklearn.cluster import Birch
from sklearn.metrics import pairwise_distances_argmin_min

def get_bae_components(scaled_features):
    feature_tensor = torch.tensor(scaled_features, dtype=torch.float32).to(device)
    with torch.no_grad():
        latent_values = bae.encoder(feature_tensor)
        reconstructed_values = bae(feature_tensor)
    
    reconstruction_scores = torch.mean((feature_tensor - reconstructed_values) ** 2, dim=1).cpu().numpy()
    latent_values = latent_values.cpu().numpy()
    return reconstruction_scores, latent_values

print("Extracting latent representations from Autoencoder...")
train_recon_scores, train_latent = get_bae_components(X_train_norm_scaled)
val_recon_scores, val_latent = get_bae_components(X_val_scaled)
test_recon_scores, test_latent = get_bae_components(X_test_scaled)

latent_scaler = StandardScaler()
train_latent_scaled = latent_scaler.fit_transform(train_latent)
val_latent_scaled = latent_scaler.transform(val_latent)
test_latent_scaled = latent_scaler.transform(test_latent)

birch_model = Birch(threshold=0.5, n_clusters=None)
birch_model.fit(train_latent_scaled)
birch_centers = birch_model.subcluster_centers_

print(f"BIRCH formed {len(birch_centers)} subclusters in the latent space")

# 3. Calculate distance to nearest BIRCH cluster center
_, train_cluster_dists = pairwise_distances_argmin_min(train_latent_scaled, birch_centers)
_, val_cluster_dists = pairwise_distances_argmin_min(val_latent_scaled, birch_centers)
_, test_cluster_dists = pairwise_distances_argmin_min(test_latent_scaled, birch_centers)

def robust_score_scale(values, reference_values):
    median = np.median(reference_values)
    spread = np.percentile(reference_values, 75) - np.percentile(reference_values, 25)
    if spread <= 1e-12:
        spread = np.std(reference_values)
        if spread <= 1e-12:
            spread = 1.0
    return (values - median) / spread

# 5. Combine: 50% Reconstruction Error + 50% BIRCH Cluster Distance
val_bae_scores = (
    robust_score_scale(val_recon_scores, train_recon_scores) +
    robust_score_scale(val_cluster_dists, train_cluster_dists)
) / 2

test_bae_scores = (
    robust_score_scale(test_recon_scores, train_recon_scores) +
    robust_score_scale(test_cluster_dists, train_cluster_dists)
) / 2

# 6. Optimize threshold on Validation PR-Curve
precision_vals, recall_vals, thresholds = precision_recall_curve(y_val, val_bae_scores)
val_f1_scores = 2 * (precision_vals[:-1] * recall_vals[:-1]) / (precision_vals[:-1] + recall_vals[:-1] + 1e-12)
best_threshold_idx = int(np.argmax(val_f1_scores))
birch_bae_threshold = thresholds[best_threshold_idx]

# 7. Evaluate on Test Set
test_birch_bae_preds = test_bae_scores >= birch_bae_threshold

print("\n BAE (BIRCH Autoencoder) Results ===")
print(f"Selected Threshold: {birch_bae_threshold:.4f}")
print(f"Test Precision:     {precision_score(y_test, test_birch_bae_preds, zero_division=0):.4f}")
print(f"Test Recall:        {recall_score(y_test, test_birch_bae_preds, zero_division=0):.4f}")
print(f"Test F1 Score:      {f1_score(y_test, test_birch_bae_preds, zero_division=0):.4f}")
print(f"Total Test Alerts:  {int(test_birch_bae_preds.sum())}")
print(f"True Positive Hits: {int((test_birch_bae_preds & y_test).sum())} / {y_test.sum()}")

Extracting latent representations from Autoencoder...
BIRCH formed 29 subclusters in the latent space

 BAE (BIRCH Autoencoder) Results ===
Selected Threshold: 0.8831
Test Precision:     0.1155
Test Recall:        0.2347
Test F1 Score:      0.1548
Total Test Alerts:  1065
True Positive Hits: 123 / 524


In [18]:
from google.colab import drive
import os
import joblib

drive.mount('/content/drive')

drive_models_dir = "/content/drive/MyDrive/models"
os.makedirs(drive_models_dir, exist_ok=True)

torch.save(bae.state_dict(), f"{drive_models_dir}/bae.pth")
print(f"Autoencoder weights saved to: {drive_models_dir}/bae.pth")

#  Compute reference stats for robust scaling at inference time
train_recon_median = float(np.median(train_recon_scores))
train_recon_iqr = float(np.percentile(train_recon_scores, 75) - np.percentile(train_recon_scores, 25))

train_cluster_median = float(np.median(train_cluster_dists))
train_cluster_iqr = float(np.percentile(train_cluster_dists, 75) - np.percentile(train_cluster_dists, 25))

bae_bundle = {
    "feature_scaler": scaler,
    "latent_scaler": latent_scaler,
    "birch_centers": birch_centers,
    "threshold": float(birch_bae_threshold),
    "feature_columns": list(X.columns),
    "stats": {
        "recon_median": train_recon_median,
        "recon_iqr": train_recon_iqr if train_recon_iqr > 1e-12 else 1.0,
        "cluster_median": train_cluster_median,
        "cluster_iqr": train_cluster_iqr if train_cluster_iqr > 1e-12 else 1.0
    }
}

joblib.dump(bae_bundle, f"{drive_models_dir}/bae_bundle.joblib")
print(f"BAE bundle saved to: {drive_models_dir}/bae_bundle.joblib")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Autoencoder weights saved to: /content/drive/MyDrive/models/bae.pth
BAE bundle saved to: /content/drive/MyDrive/models/bae_bundle.joblib
